# Práctica 2: Predicción de la Demanda Eléctrica en España

## Planteamiento del problema

Red Eléctrica de España (REE) es el operador del sistema eléctrico. Publica de forma abierta, a través de su servicio REData, la demanda de energía, los precios del mercado y la generación por tecnología. Una predicción fiable de la demanda a una semana vista es útil para planificar qué centrales se programan, para comprar energía con antelación y para anticipar situaciones de tensión en la red.
En este trabajo, tienes que construir y evaluar un modelo que prediga la demanda eléctrica diaria para los próximos 7 días.

## Requisitos técnicos e instalación de librerías necesarias

Para reproducir el proyecto se recomienda crear un entorno virtual e instalar las versiones exactas de `requirements.txt`:

```bash
python -m venv .venv
# Windows: .venv\Scripts\activate | Linux/Mac: source .venv/bin/activate
pip install -r requirements.txt
```

## Implementación

### Librerías a importar

In [1]:
# Librerías generales
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Análisis exploratorio

# 2. Preprocesado de datos

# 3. Selección de variables

# 4. Modelos (skforecast)

# 5. Evaluación y métricas

# 6. Análisis estadísticos

# Fijamos la semilla para que los resultados sean reproducibles
SEED = 42
np.random.seed(SEED)

# Configuración de estilo para los gráficos
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

### Carga de la base de datos (csv)

In [2]:
# Cargamos el dataset de demanda y precios, una fila por hora
original_ree_hourly = pd.read_csv("data/ree_horario.csv", encoding="UTF-8")

print(f"Dimensiones del dataset horario: {original_ree_hourly.shape}")
# Mostramos las primeras 5 filas
original_ree_hourly.head()

Dimensiones del dataset horario: (26280, 6)


,fecha_hora,demanda_prevista,demanda_programada,demanda_real,precio_spot,pvpc
0,2021-01-01 00:00:00+01:00,26104.667,26013.0,26151.667,50.87,NaN
1,2021-01-01 01:00:00+01:00,24822.583,24630.0,24886.250,48.19,NaN
2,2021-01-01 02:00:00+01:00,22652.667,22696.0,22685.500,44.68,NaN
3,2021-01-01 03:00:00+01:00,20786.833,20756.0,20828.250,38.50,NaN
4,2021-01-01 04:00:00+01:00,19877.917,19800.0,19877.583,36.80,NaN


In [3]:
# Cargamos el dataset de generación por tecnología: una fila por día
original_ree_daily_generation = pd.read_csv("data/ree_generacion_diaria.csv", encoding="UTF-8")

print(f"Dimensiones del dataset de generación diaria: {original_ree_daily_generation.shape}")
# Mostramos las primeras 5 filas
original_ree_daily_generation.head()

Dimensiones del dataset de generación diaria: (1095, 18)


,fecha,gen_hidráulica,gen_nuclear,gen_carbón,gen_motores_diésel,gen_turbina_de_gas,gen_turbina_de_vapor,gen_ciclo_combinado,gen_hidroeólica,gen_eólica,gen_solar_fotovoltaica,gen_solar_térmica,gen_otras_renovables,gen_cogeneración,gen_residuos_no_renovables,gen_residuos_renovables,gen_generación_total,gen_fuel_gas
0,2021-01-01,111077.579,170966.633,5663.327,6180.245,840.614,3681.076,61416.329,15.538,240290.824,29876.523,3126.095,13234.386,54581.986,4477.651,1232.274,706661.080,NaN
1,2021-01-02,112211.332,170974.655,5611.373,7074.243,722.388,4587.264,80555.288,1.957,270599.158,34247.163,6340.881,12636.823,67513.021,4389.159,1262.089,778726.794,NaN
2,2021-01-03,116969.983,171000.851,5597.198,7036.465,993.803,4502.216,83628.611,15.593,221167.693,37134.485,7441.714,12215.001,73185.396,4896.856,1250.171,747036.037,NaN
3,2021-01-04,165965.559,170871.508,8431.686,7371.259,1328.929,3783.796,144897.990,34.367,130542.499,26307.981,2783.566,12124.513,77455.663,5155.834,1215.017,758270.167,NaN
4,2021-01-05,176184.581,170882.098,16295.678,7285.896,1006.802,4453.566,138876.901,-3.059,70360.924,29746.352,2099.695,12386.598,80838.906,5499.507,1230.110,717144.555,NaN


### 1. Análisis exploratorio de datos

#### 1.1 Estructura general de los datasets

Revisamos qué contiene cada fichero tal y como lo hemos cargado en un inicio: número de filas, tipo de cada variable y valores no nulos. Agrupamos las variables según el enunciado: demanda (MW) y precios (€/MWh) en el fichero horario, y generación por tecnología (MWh) en el diario, porque cada grupo tiene unidades y significado distintos. En este apartado solo consultamos los datos, por lo que trabajamos directamente con los originales.

In [4]:
# Definimos nuestra variable objetivo
target = "demanda_real"
# Definimos las variables de cada dataset según el enunciado
demand_features = ["demanda_real", "demanda_prevista", "demanda_programada"]
price_features = ["precio_spot", "pvpc"]
generation_features = [gen for gen in original_ree_daily_generation.columns if gen != "fecha"]

# Comprobamos que no se nos queda ninguna variable fuera (sumamos 1 por la columna fecha)
assert len(demand_features) + len(price_features) + 1 == original_ree_hourly.shape[1]
assert len(generation_features) + 1 == original_ree_daily_generation.shape[1]

# Observamos la info de cada dataset
print(f"Demanda: {len(demand_features)} | Precios: {len(price_features)} | Generación: {len(generation_features)}")
original_ree_hourly.info()
original_ree_daily_generation.info()

Demanda: 3 | Precios: 2 | Generación: 17
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26280 entries, 0 to 26279
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   fecha_hora          26280 non-null  object 
 1   demanda_prevista    26172 non-null  float64
 2   demanda_programada  26172 non-null  float64
 3   demanda_real        26178 non-null  float64
 4   precio_spot         26280 non-null  float64
 5   pvpc                22657 non-null  float64
dtypes: float64(5), object(1)
memory usage: 1.2+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1095 entries, 0 to 1094
Data columns (total 18 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   fecha                       1095 non-null   object 
 1   gen_hidráulica              1095 non-null   float64
 2   gen_nuclear                 1095 non-null   float64
 3   gen_carbón

**Observación:** El fichero horario tiene 26280 filas y el diario 1095, lo que corresponde a 3 años completos `(3 × 365 = 1095 días y 1095 × 24 = 26280 horas)`. Todas las variables son numéricas (`float64`) salvo las columnas de fecha (`fecha_hora` y `fecha`), que pandas interpreta como `object`. Para analizar las series necesitamos convertirlas en un índice temporal, lo cual lo haremos en el siguiente apartado. Los recuentos de no nulos ya muestran valores ausentes en cada dataset (en `original_ree_hourly` todas las variables a excepción de `fecha_hora` y `precio_spot`, y en `original_ree_daily_generation` únicamente la variable `gen_fuel_gas`). Los ficheros tienen frecuencias distintas (horaria y diaria) y unidades distintas (MW, €/MWh y MWh).

#### 1.2 Índice temporal

Para trabajar con series temporales, necesitamos que la fecha sea el índice, con tipo fecha y comprobar que no falta ni sobra ningún instante. La columna `fecha_hora` incluye el desfase respecto a UTC, que cambia entre el horario de invierno (+01:00) y el de verano (+02:00). Si lo ignorásemos, en los cambios de hora de octubre las 02:00 aparecerían duplicadas y en los de marzo faltarían, y pandas no puede construir un índice de fechas con dos desfases distintos. Por eso convertimos primero a UTC, donde cada instante es único, y después a hora peninsular (`Europe/Madrid`), porque el día natural español es el que marca el ciclo de la demanda y el que usa el fichero de generación. Comprobamos después que ambos índices son continuos.

In [5]:
# Partimos de los datasets originales para no generar errores a la hora de ejecutar la celda varias veces

# 1) Dataset horario (ree_hourly)

# Observamos los desfases respecto a UTC presentes en la columna fecha_hora (últimos 6 caracteres)
print(original_ree_hourly["fecha_hora"].str[-6:].value_counts())

# Eliminamos la columna original de fecha_hora porque la vamos a usar como índice convertida a UTC y posteriormente a hora peninsular
ree_hourly = original_ree_hourly.drop(columns="fecha_hora")
# Convertimos a UTC (instante único) y después a hora peninsular (así ya la tenemos en formato fecha)
ree_hourly.index = pd.to_datetime(original_ree_hourly["fecha_hora"], utc=True).dt.tz_convert("Europe/Madrid")
ree_hourly.index.name = "fecha_hora"
# Mostramos ahora el tipo del índice (debe ser un datetime)
print(f"Índice horario: {ree_hourly.index.dtype}")

# 2) Dataset diario (ree_daily_generation)

# La columna fecha ya es el día natural. En este caso, solo la convertimos y la usamos como índice
ree_daily_generation = original_ree_daily_generation.drop(columns="fecha")
ree_daily_generation.index = pd.to_datetime(original_ree_daily_generation["fecha"])
ree_daily_generation.index.name = "fecha"
# Mostramos ahora el tipo del índice (debe ser un datetime)
print(f"Índice diario:  {ree_daily_generation.index.dtype}")

# 3) Comprobamos que no hay instantes duplicados ni ausentes
expected_hours = pd.date_range(ree_hourly.index.min(), ree_hourly.index.max(), freq="h")
# No hace falta poner la frecuencia en este caso porque por defecto es 'D'
expected_days = pd.date_range(ree_daily_generation.index.min(), ree_daily_generation.index.max())
# Los cambios de hora no deben salir duplicados, porque aunque tenga la misma hora local, son instantes distintos (eso es lo que comprobamos aquí)
print(f"Dataset horario: duplicados: {ree_hourly.index.duplicated().sum()} | Horas faltantes: {len(expected_hours.difference(ree_hourly.index))}")
print(f"Dataset diario: duplicados: {ree_daily_generation.index.duplicated().sum()} | Días faltantes: {len(expected_days.difference(ree_daily_generation.index))}")

# Obtenemos los días que tienen cambio de hora (tendrán 23 o 25)
hours_per_day = ree_hourly.groupby(ree_hourly.index.date).size()
hours_per_day[hours_per_day != 24].rename("Horas")



fecha_hora
+02:00    15624
+01:00    10656
Name: count, dtype: int64
Índice horario: datetime64[ns, Europe/Madrid]
Índice diario:  datetime64[ns]
Dataset horario: duplicados: 0 | Horas faltantes: 0
Dataset diario: duplicados: 0 | Días faltantes: 0


2021-03-28    23
2021-10-31    25
2022-03-27    23
2022-10-30    25
2023-03-26    23
2023-10-29    25
Name: Horas, dtype: int64

**Observación:** En el dataset horario, en la columna `fecha_hora` conviven 15624 horas con desfase +02:00 (horario de verano) y 10656 con desfase +01:00 (horario de invierno). Tras la conversión de ambos datasets, los 2 índices son de tipo fecha (`datetime64[ns, Europe/Madrid]` el horario y `datetime64[ns]` el diario). El dataset diario, al tratarse de días naturales, no necesita zona horaria. Ambos son continuos, sin instantes duplicados ni ausentes. Los cambios de hora no aparecen como duplicados porque, con la zona horaria, son instantes distintos separados por una hora.

En hora peninsular todos los días tienen 24 horas salvo los 6 de cambio de hora (3 con 23 horas y 3 con 25). Como se compensan, el total sigue siendo `1095 × 24 = 26280 horas`. Esto habrá que tenerlo en cuenta más adelante al pasar la demanda a frecuencia diaria. 

#### 1.3 Valores nulos

Cuantificamos los valores ausentes de cada variable y analizamos cómo se distribuyen en el tiempo. En una serie temporal no basta con contar los nulos, ya que no es lo mismo una hora suelta, que se puede reconstruir a partir de las vecinas, que un bloque largo de horas seguidas o un periodo entero sin datos. Para cada variable con nulos calculamos el número y porcentaje de valores ausentes, días afectados, el bloque más largo de nulos consecutivos y el periodo en el que aparecen. Para esto, vamos a definir varias funciones para poder usarlas más adelante.

In [6]:
def obtain_longest_missing_block(series):
    """
    Calcula la longitud del bloque más largo de valores nulos consecutivos de una serie.

    Cada valor no nulo abre un bloque nuevo y los nulos que le siguen se asignan a ese bloque,
    de modo que el número de nulos de cada bloque es la longitud de una racha de nulos seguidos.
    La longitud se expresa en pasos de la serie: horas en el dataset horario y días en el diario.

    Parámetros
    ----------
    series : pd.Series
        Serie ordenada temporalmente en la que se buscan los nulos.

    Returns
    -------
    int
        Número de valores nulos del bloque consecutivo más largo (0 si la serie no tiene nulos).
    """
    is_missing = series.isna()
    # Cada valor no nulo abre un bloque nuevo y los nulos que le siguen pertenecerán a dicho bloque
    block_id = (~is_missing).cumsum()
    return int(is_missing.groupby(block_id).sum().max())

def missing_summary(df, unit):
    """
    Resume los valores nulos de cada variable de un DataFrame con índice temporal.

    Para cada variable calcula el número y el porcentaje de nulos, los días naturales afectados,
    el bloque más largo de nulos consecutivos y el primer y último instante con nulos.
    Solo se devuelven las variables que tienen algún nulo.

    Parámetros
    ----------
    df : pd.DataFrame
        Datos con un DatetimeIndex ordenado y continuo (horario o diario).
    unit : str
        Unidad de los pasos del índice ("horas" o "días"), usada en el nombre de la
        columna del bloque más largo.

    Returns
    -------
    pd.DataFrame
        Una fila por variable con nulos y las columnas "Nulos", "%", "Días afectados",
        "Bloque más largo ({unit})", "Primer nulo" y "Último nulo".
    """
    summary = pd.DataFrame({
        "Nulos": df.isna().sum(),
        "%": (df.isna().mean() * 100).round(2),
        "Días afectados": [pd.Series(df.index[df[col].isna()].date).nunique() for col in df.columns],
        f"Bloque más largo ({unit})": [obtain_longest_missing_block(df[col]) for col in df.columns],
        "Primer nulo": [df.index[df[col].isna()].min() for col in df.columns],
        "Último nulo": [df.index[df[col].isna()].max() for col in df.columns],
    })
    # Únicamente mostramos las variables que tienen algún nulo
    return summary[summary["Nulos"] > 0]

# Mostramos el resumen de nulos para cada dataset
display(missing_summary(ree_hourly, "horas"))
display(missing_summary(ree_daily_generation, "días"))

,Nulos,%,Días afectados,Bloque más largo (horas),Primer nulo,Último nulo
demanda_prevista,108,0.41,6,18,2021-05-22 03:00:00+02:00,2023-10-03 20:00:00+02:00
demanda_programada,108,0.41,6,18,2021-05-22 03:00:00+02:00,2023-10-03 20:00:00+02:00
demanda_real,102,0.39,6,17,2021-05-22 04:00:00+02:00,2023-11-05 20:00:00+01:00
pvpc,3623,13.79,151,3623,2021-01-01 00:00:00+01:00,2021-05-31 23:00:00+02:00


,Nulos,%,Días afectados,Bloque más largo (días),Primer nulo,Último nulo
gen_fuel_gas,1085,99.09,1085,371,2021-01-01,2023-12-31


**Observación:** los valores ausentes son de dos tipos muy distintos.

- **Variables sin datos en un periodo o prácticamente vacías.** `pvpc` no tiene ningún valor entre el 1 de enero y el 31 de mayo de 2021 (3623 horas, el 13,79 %), todas en un único bloque, y a partir del 1 de junio de 2021 está completa. Si investigamos un poco, justo a partir del 1 de junio de 2021, entró en vigor la estructura actual de peajes 2.0TD, así que no lo consideramos como un fallo puntual: la serie, con esta definición, no existe antes de esa fecha. `gen_fuel_gas` está vacía en 1085 de los 1095 días (99,09 %).
- **Bloques largos en las series de demanda.** `demanda_real` (102 horas, 0,39 %) y `demanda_prevista` y `demanda_programada` (108 horas cada una, 0,41 %) tienen muy pocos nulos en proporción, pero concentrados en solo 6 días por serie, en bloques de 17 y 18 horas seguidas. No son horas sueltas que se puedan reconstruir a partir de sus vecinas.

Analizamos con más detalle los días con nulos en la demanda, que incluye la variable objetivo, y los pocos valores que tiene `gen_fuel_gas`.

In [7]:
# Obtenemos las horas sin dato en cada serie de demanda, agrupadas por día natural
demand_missing = ree_hourly[demand_features].isna()
missing_by_day = demand_missing.groupby(ree_hourly.index.date).sum()
# Nos quedamos solo con los días que tienen algún nulo en alguna de las tres series
missing_by_day = missing_by_day[missing_by_day.sum(axis=1) > 0]

# Franja horaria afectada en cada uno de esos días (primera y última hora con nulos)
missing_hours = ree_hourly.index[demand_missing.any(axis=1)]
missing_by_day["Franja con nulos"] = (pd.Series(missing_hours.hour, index=missing_hours.date).groupby(level=0).agg(lambda hours: f"{hours.min():02d}-{hours.max():02d} h"))
display(missing_by_day)

# Valores no nulos de gen_fuel_gas
ree_daily_generation["gen_fuel_gas"].dropna()

,demanda_real,demanda_prevista,demanda_programada,Franja con nulos
2021-05-22,17,18,18,03-20 h
2021-11-07,17,0,0,04-20 h
2022-03-06,0,18,18,03-20 h
2022-06-29,0,18,18,03-20 h
2022-07-27,17,0,0,04-20 h
2022-12-13,17,0,0,04-20 h
2022-12-19,0,18,18,03-20 h
2023-04-12,0,18,18,03-20 h
2023-04-18,17,0,0,04-20 h
2023-10-03,0,18,18,03-20 h


fecha
2021-07-25   -0.001
2021-08-08    0.001
2021-08-19   -0.001
2021-09-12    0.001
2021-09-18   -0.001
2022-09-25   -0.001
2022-09-29    0.001
2022-10-20   -0.001
2022-10-27    0.001
2023-07-22    0.001
Name: gen_fuel_gas, dtype: float64

**Observación:** hay 11 días con algún hueco en la demanda, y siguen un patrón bastante regular. En `demanda_real` falta siempre la franja de 04 a 20 h (17 horas), y en `demanda_prevista` y `demanda_programada` la de 03 a 20 h (18 horas). Estas dos últimas faltan siempre a la vez. Solo el 2021-05-22 tiene nulos en las tres variables. Que el patrón se repita de forma tan exacta nos da a pensar que apunta a fallos de descarga o publicación de la API, y no a que la demanda no se midiera. Por otro lado, los 10 valores de `gen_fuel_gas` son ±0,001 MWh, es decir, casi 0. Esta variable no aporta ninguna información.

El caso de `demanda_real` es el más delicado porque es la variable objetivo. En esos 6 días solo quedan 7 horas, de 21h a 03h. Si calculáramos la media diaria con las horas disponibles, seguramente esos días tendrían una demanda baja (últimas horas del día, incluyendo madrugada). Con huecos de 17 horas tampoco tiene sentido una interpolación lineal entre la hora anterior y la posterior, porque se perdería todo el ciclo diario. Decidimos cómo tratarlo en el preprocesado.

#### 1.4 Estadísticos descriptivos

En este apartado calculamos los estadísticos básicos de cada variable y añadimos la asimetría (`skew`) para detectar valores imposibles, comparando mínimos y máximos con lo que es físicamente esperable, y para identificar distribuciones con colas largas. En este problema todas las variables son magnitudes que, en principio, no deberían ser negativas (hablamos de potencias, energías y precios), así que revisamos además los valores negativos y los que sean 0.

In [8]:
# 1) Estadísticos de las variables horarias (demanda y precios)
hourly_summary = ree_hourly.describe().T
# Añadimos la asimetría (skewness) de cada variable
hourly_summary["skew"] = ree_hourly.skew()
display(hourly_summary.round(2))

# 2) Estadísticos de la generación diaria por tecnología
generation_summary = ree_daily_generation.describe().T
generation_summary["skew"] = ree_daily_generation.skew()
display(generation_summary.round(2))

# La Skew la interpretamos de la siguiente manera:
# - Skew = 0 => distribución simétrica
# - Skew > 0 => distribución sesgada a la derecha. La mayoría de los valores son moderados y unos pocos muy altos
# - Skew < 0 => distribución sesgada a la izquierda. Caso contrario al anterior, pocos valores muy bajos

,count,mean,std,min,25%,50%,75%,max,skew
demanda_prevista,26172.0,28645.19,4569.18,17165.00,24901.52,28754.46,32096.19,43862.75,0.09
demanda_programada,26172.0,28654.59,4598.18,17079.00,24875.81,28783.00,32125.62,43711.00,0.08
demanda_real,26178.0,28680.79,4573.27,17199.42,24938.85,28791.79,32136.17,43893.92,0.09
precio_spot,26280.0,122.18,71.89,0.00,75.12,110.00,163.82,700.00,1.00
pvpc,22657.0,217.19,105.41,11.68,140.25,202.37,281.34,954.01,0.85


,count,mean,std,min,25%,50%,75%,max,skew
gen_hidráulica,1095.0,66942.36,37301.06,12758.18,38984.56,56974.89,84145.15,191711.08,1.11
gen_nuclear,1095.0,150340.23,20694.88,78628.55,141400.98,160035.90,166872.54,171217.82,-0.94
gen_carbón,1095.0,15298.39,8216.68,-643.44,8391.24,13149.29,19812.14,44393.35,0.83
gen_motores_diésel,1095.0,6964.70,962.54,4692.08,6273.34,6965.66,7614.38,9924.46,0.16
gen_turbina_de_gas,1095.0,1682.47,869.11,143.82,1017.02,1549.84,2215.89,4798.72,0.67
gen_turbina_de_vapor,1095.0,3260.44,888.25,563.57,2837.92,3315.04,3852.38,5795.69,-0.41
gen_ciclo_combinado,1095.0,145497.55,77672.84,33657.93,85450.95,129223.33,184177.50,396453.64,0.97
gen_hidroeólica,1095.0,58.30,45.20,-4.07,16.37,49.58,96.01,155.10,0.43
gen_eólica,1095.0,168395.10,87870.72,20866.06,102053.07,151098.20,221158.97,430064.38,0.74
gen_solar_fotovoltaica,1095.0,78850.10,35776.75,6751.87,51233.75,75118.44,106190.64,162950.84,0.31


Como hemos comentado anteriormente, también revisamos los valores negativos y los ceros de cada variable, y a qué periodo pertenecen. Vamos a excluir la variable `gen_fuel_gas`, que ya hemos visto que es prácticamente nula.

In [9]:
def non_positive_summary(df):
    """
    Resume los valores negativos y los ceros de cada variable de un DataFrame con índice temporal.

    Solo se devuelven las variables que tienen algún valor negativo o igual a cero.

    Parámetros
    ----------
    df : pd.DataFrame
        Datos con un DatetimeIndex.

    Returns
    -------
    pd.DataFrame
        Una fila por variable afectada y las columnas "Negativos", "Ceros", "Mínimo",
        "Primer caso" y "Último caso" (primer y último instante con un valor <= 0).
    """
    summary = pd.DataFrame({
        "Negativos": (df < 0).sum(),
        "Ceros": (df == 0).sum(),
        "Mínimo": df.min(),
        "Primer caso": [df.index[df[col] <= 0].min() for col in df.columns],
        "Último caso": [df.index[df[col] <= 0].max() for col in df.columns],
    })
    # Únicamente mostramos las variables que tienen algún valor negativo o cero
    return summary[(summary["Negativos"] + summary["Ceros"]) > 0]

# Mostramos resumen de valores negativos y ceros para cada dataset
display(non_positive_summary(ree_hourly))
display(non_positive_summary(ree_daily_generation.drop(columns="gen_fuel_gas")))

# Horas con precio spot igual a cero, por año y mes
spot_zero_hours = ree_hourly.index[ree_hourly["precio_spot"] == 0]
display(pd.crosstab(spot_zero_hours.year, spot_zero_hours.month, rownames=["Año"], colnames=["Mes"]))

# Comprobamos si la generación total es la suma de todas las tecnologías
technology_features = [col for col in generation_features if col != "gen_generación_total"]
total_difference = (ree_daily_generation[technology_features].sum(axis=1) - ree_daily_generation["gen_generación_total"]).abs()
print(f"Diferencia máxima entre la suma de tecnologías y gen_generación_total: {total_difference.max():.3f} MWh")

,Negativos,Ceros,Mínimo,Primer caso,Último caso
precio_spot,0,112,0.0,2022-12-31 03:00:00+01:00,2023-12-31 05:00:00+01:00


,Negativos,Ceros,Mínimo,Primer caso,Último caso
gen_carbón,5,0,-643.438,2021-01-23,2021-02-07
gen_hidroeólica,31,0,-4.074,2021-01-05,2022-12-25


Mes,1,3,4,5,6,7,8,9,10,11,12
Año,,,,,,,,,,,
2022,0,0,0,0,0,0,0,0,0,0,3
2023,13,8,19,12,1,12,12,4,7,19,2


Diferencia máxima entre la suma de tecnologías y gen_generación_total: 0.001 MWh


**Observación:** las tres series de demanda tienen una distribución casi simétrica (`skew` ≈ 0,09) y valores físicamente razonables, entre unos 17100 MW y 43900 MW, con una media de unos 28700 MW. Los estadísticos de la demanda real, la prevista y la programada son prácticamente iguales, es decir, las tres series tienen una distribución muy parecida.

Los precios tienen asimetría positiva (`skew` 1,00 en `precio_spot` y 0,85 en `pvpc`), lo que indica que la mayoría de las horas tienen precios moderados y unas pocas, precios muy altos, con máximos de 700 €/MWh en el spot y 954 €/MWh en el PVPC. Además, `precio_spot` vale exactamente 0 en 112 horas (entre diciembre de 2022 y diciembre de 2023). Un precio de 0 €/MWh no es imposible en un mercado eléctrico, por lo que no lo consideramos un error, pero los ceros descartarían transformaciones como el logaritmo si llegásemos a usar el precio.

En la generación aparecen valores negativos en `gen_carbón` (5 días entre el 23 de enero y el 7 de febrero de 2021, con un mínimo de -643 MWh) y en `gen_hidroeólica` (31 días entre enero de 2021 y diciembre de 2022, con un mínimo de -4 MWh). Una energía generada negativa no tiene sentido como generación, así que habrá que decidir cómo tratarlos si se usan estas variables. También comprobamos que `gen_generación_total` coincide con la suma de todas las tecnologías (la diferencia máxima es de 0,001 MWh), por lo que es una variable redundante con el resto. Por último, las escalas de las variables son muy distintas (de decenas a cientos de miles de MWh) y algunas tecnologías tienen asimetrías marcadas, positivas (`gen_hidráulica` 1,11, `gen_ciclo_combinado` 0,97) o negativas (`gen_residuos_renovables` -1,29, `gen_nuclear` y `gen_otras_renovables` -0,94).

### 2. Preprocesamiento de datos

### 3. Selección de variables

### 4. Modelado con skforecast

### 5. Evaluación

### 6. Análisis estadísticos (opcional)

## Conclusiones